# Taller de código 3: Q-learning

Entrenarás un agente en un pequeño laberinto inspirado en Pac-Man. Debe llegar al punto de comida, evitar al fantasma y aprender únicamente mediante interacción.

Tiempo estimado: **75–90 minutos**.

## Entorno

- `S`: estado inicial.
- `G`: comida/meta, recompensa `+10` y fin del episodio.
- `X`: fantasma, recompensa `-10` y fin del episodio.
- `#`: pared.
- cada movimiento normal cuesta `-0.1`.

El entorno es determinístico para concentrarnos en la actualización de Q-learning.

In [ ]:
from collections import defaultdict
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

ROWS, COLS = 4, 5
START = (3, 0)
GOAL = (0, 4)
GHOST = (1, 3)
WALLS = {(1, 1), (2, 1), (2, 3)}
ACTIONS = ["up", "down", "left", "right"]
DELTA = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}

In [ ]:
def render(state=None):
    symbols = []
    for r in range(ROWS):
        row = []
        for c in range(COLS):
            s = (r, c)
            if s == state: row.append("P")
            elif s == START: row.append("S")
            elif s == GOAL: row.append("G")
            elif s == GHOST: row.append("X")
            elif s in WALLS: row.append("#")
            else: row.append("·")
        symbols.append(row)
    return pd.DataFrame(symbols)

render(START)

## Actividad 1 — Dinámica del entorno

In [ ]:
def step(state, action):
    """Devuelve next_state, reward, done."""
    dr, dc = DELTA[action]
    candidate = (state[0] + dr, state[1] + dc)

    if not (0 <= candidate[0] < ROWS and 0 <= candidate[1] < COLS):
        candidate = state
    if candidate in WALLS:
        candidate = state

    if candidate == GOAL:
        return candidate, 10.0, True
    if candidate == GHOST:
        return candidate, -10.0, True
    return candidate, -0.1, False

In [ ]:
assert step((0, 3), "right") == (GOAL, 10.0, True)
assert step((1, 2), "right") == (GHOST, -10.0, True)
assert step(START, "left")[0] == START
assert step((2, 0), "right")[0] == (2, 0)  # pared
print("Entorno validado.")

## Actividad 2 — Política $\epsilon$-greedy

Con probabilidad $\epsilon$, explora una acción aleatoria. En caso contrario, escoge una acción con valor Q máximo. Desempata aleatoriamente.

In [ ]:
Q = defaultdict(float)

def epsilon_greedy(Q, state, epsilon):
    if random.random() < epsilon:
        return random.choice(ACTIONS)
    q_vals = [Q[(state, a)] for a in ACTIONS]
    max_q = max(q_vals)
    best_actions = [a for a, q in zip(ACTIONS, q_vals) if q == max_q]
    return random.choice(best_actions)

In [ ]:
random.seed(RANDOM_STATE)
Q_test = defaultdict(float, {((3, 0), "right"): 5.0})
assert epsilon_greedy(Q_test, (3, 0), epsilon=0.0) == "right"
assert epsilon_greedy(Q_test, (3, 0), epsilon=1.0) in ACTIONS
print("Pruebas epsilon-greedy superadas.")

## Actividad 3 — Actualización de Q-learning

Implementa:

$$Q(s,a)\leftarrow Q(s,a)+\alpha\left[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\right].$$

Si `done=True`, no existe recompensa futura y el target es solamente $r$.

In [ ]:
def update_q(Q, state, action, reward, next_state, done, alpha, gamma):
    if done:
        td_target = reward
    else:
        best_next_q = max(Q[(next_state, a)] for a in ACTIONS)
        td_target = reward + gamma * best_next_q
    td_error = td_target - Q[(state, action)]
    Q[(state, action)] += alpha * td_error
    return td_target, td_error

In [ ]:
Q_test = defaultdict(float)
update_q(Q_test, (0, 3), "right", 10, GOAL, True, alpha=0.5, gamma=0.9)
assert np.isclose(Q_test[((0, 3), "right")], 5.0)

Q_test = defaultdict(float, {((2, 0), "up"): 2.0})
update_q(Q_test, START, "up", -0.1, (2, 0), False, alpha=0.5, gamma=0.9)
assert np.isclose(Q_test[(START, "up")], 0.85)
print("Pruebas de actualización superadas.")

## Actividad 4 — Entrenamiento

In [ ]:
def train_q_learning(episodes=3000, alpha=0.2, gamma=0.95,
                     epsilon_start=1.0, epsilon_end=0.05,
                     max_steps=100):
    Q = defaultdict(float)
    episode_rewards = []
    
    for ep in range(episodes):
        state = START
        total_reward = 0
        epsilon = epsilon_start - (ep / max(1, episodes - 1)) * (epsilon_start - epsilon_end)
        
        for _ in range(max_steps):
            action = epsilon_greedy(Q, state, epsilon)
            next_state, reward, done = step(state, action)
            update_q(Q, state, action, reward, next_state, done, alpha, gamma)
            
            total_reward += reward
            state = next_state
            if done:
                break
                
        episode_rewards.append(total_reward)
        
    return Q, episode_rewards

In [ ]:
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
Q, episode_rewards = train_q_learning()

assert len(episode_rewards) == 3000
print("Promedio primeras 100:", np.mean(episode_rewards[:100]))
print("Promedio últimas 100:", np.mean(episode_rewards[-100:]))

## Actividad 5 — Curva de aprendizaje

In [ ]:
rewards = pd.Series(episode_rewards)
plt.plot(rewards.rolling(100).mean())
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio móvil")
plt.title("Aprendizaje del agente")
plt.show()

## Actividad 6 — Política aprendida

In [ ]:
def greedy_policy(Q):
    policy = {}
    for r in range(ROWS):
        for c in range(COLS):
            s = (r, c)
            if s not in WALLS and s != GOAL and s != GHOST:
                q_vals = [Q[(s, a)] for a in ACTIONS]
                max_q = max(q_vals)
                best_actions = [a for a, q in zip(ACTIONS, q_vals) if q == max_q]
                policy[s] = random.choice(best_actions)
    return policy


def show_policy(policy):
    arrows = {"up": "↑", "down": "↓", "left": "←", "right": "→"}
    grid = []
    for r in range(ROWS):
        row = []
        for c in range(COLS):
            s = (r, c)
            if s in WALLS: row.append("#")
            elif s == GOAL: row.append("G")
            elif s == GHOST: row.append("X")
            else: row.append(arrows[policy[s]])
        grid.append(row)
    return pd.DataFrame(grid)

policy = greedy_policy(Q)
show_policy(policy)

## Actividad 7 — Ejecutar la política

In [ ]:
def run_greedy_episode(Q, max_steps=30):
    state = START
    trajectory = [state]
    total_reward = 0
    
    for _ in range(max_steps):
        q_vals = [Q[(state, a)] for a in ACTIONS]
        max_q = max(q_vals)
        best_actions = [a for a, q in zip(ACTIONS, q_vals) if q == max_q]
        action = best_actions[0]
        
        next_state, reward, done = step(state, action)
        trajectory.append(next_state)
        total_reward += reward
        state = next_state
        if done:
            break
            
    return trajectory, total_reward

trajectory, total_reward = run_greedy_episode(Q)
print("Trayectoria:", trajectory)
print("Recompensa:", total_reward)
assert trajectory[0] == START
assert trajectory[-1] in {GOAL, GHOST}

## Preguntas de análisis y respuestas

1. **Compara $Q((1,2),\text{right})$ —acción hacia el fantasma— con las otras acciones del mismo estado. ¿Qué esperas observar?**
   - Se observa un valor $Q((1,2),\text{right})$ fuertemente negativo (penalizado con recompensa inmediata de $-10.0$ por colisión con el fantasma), mientras que las acciones que alejan al agente (como `up` o `down`) convergen a valores $Q$ positivos al conducir con seguridad hacia la meta `GOAL` ($+10.0$).

2. **¿Qué ocurre si entrenas con $\epsilon=0$ desde el primer episodio?**
   - El agente solo explota el conocimiento inicial ($Q=0$). Toma la primera acción arbitraria y queda atrapado en ciclos o políticas miopes sin explorar el espacio de estados, fracasando en descubrir la meta o las rutas óptimas.

3. **¿Qué efecto tendría reducir $\gamma$ casi hasta cero?**
   - El agente se vuelve completamente miope (egoísta a corto plazo), ignorando recompensas futuras ($\gamma 	o 0 \implies 	ext{target} pprox r$). Solo perseguiría evitar la penalización inmediata por paso ($-0.1$) o el fantasma, perdiendo la motivación para planear rutas de varios pasos hacia `GOAL`.

4. **¿Por qué una recompensa alta en un episodio no demuestra por sí sola que el agente aprendió?**
   - Un episodio aislado puede alcanzar la meta por pura suerte o exploración aleatoria estocástica ($\epsilon$). El aprendizaje genuino se demuestra estadísticamente cuando la curva de recompensa promedio móvil se estabiliza consistentemente en un nivel alto y la política greedy reproduce trayectorias óptimas repetidamente.

5. **Convierte el entorno en estocástico: con probabilidad 0.1 ejecuta una acción distinta. ¿Qué cambia en el aprendizaje?**
   - El agente aprende políticas más conservadoras y aversas al riesgo, alejándose de los bordes o de celdas adyacentes al fantasma $X$, pues una acción resbaladiza inesperada podría enviarlo fatalmente contra él. La convergencia requiere mayor número de episodios y una tasa $lpha$ decayente.

## Checklist

- [x] Implementé $\epsilon$-greedy.
- [x] Implementé la actualización de Q-learning.
- [x] Entrené durante varios episodios.
- [x] Visualicé la curva de aprendizaje.
- [x] Extraje y ejecuté la política greedy.